In [2]:
import os
import sys
import numpy as np
import pandas as pd
from collections import Counter

import matplotlib.pyplot as plt
import seaborn as sns
from scipy.signal import savgol_filter
from sklearn.model_selection import train_test_split

import scanpy as sc
import anndata

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    HfArgumentParser,
    TrainingArguments,
    Trainer,
    default_data_collator,
)
from peft import (
    LoraConfig,
    PeftConfig,
    PeftModel,
    get_peft_model,
)

from dataclasses import dataclass, field
from typing import Optional
from enum import Enum
import evaluate

sys.path.append("..")
from lib_vqvae.vqvae import VQVAE
from lib_vqvae.dataset import *

#### Load the pre-trained cell tokenizer

In [3]:
import warnings
warnings.filterwarnings("ignore")
import argparse

parser = argparse.ArgumentParser()
parser.add_argument("--train_batch_size", type=int, default=64)
parser.add_argument("--eval_batch_size", type=int, default=2000)
parser.add_argument("--encoder_hidden_dim", type=list, default=[1600,1024,800])
parser.add_argument("--decoder_hidden_dim", type=list, default=[800,1024,1600])
parser.add_argument("--codebook_dim", type=int, default=8)
parser.add_argument("--n", type=int, default=32)
parser.add_argument("--n_codebooks", type=int, default=256)
parser.add_argument("--gamma", type=float, default=5)
parser.add_argument("--beta", type=float, default=10)
parser.add_argument('--weight_decay', type=float, default=5e-4)
parser.add_argument("--lr", type=float, default=1e-4) # 1e-3
parser.add_argument('--max_iter', type=int, default=20)
parser.add_argument('--seed', type=int, default=2024)
parser.add_argument('--gpu', type=int, default=0)
parser.add_argument("--dropout_rate", type=float, default=0.2)
parser.add_argument('--mode', type=str, default="vqvae")
parser.add_argument('--hvg', type=int, default=5000)

args = parser.parse_args([])

# -- device
if torch.cuda.is_available():
    args.device='cuda'
    torch.cuda.set_device(args.gpu)
else:
    args.device='cpu' 
print(args.device)
    
data_dim = 5000
vae = VQVAE(data_dim, args.n, args.codebook_dim, args.n_codebooks, args.encoder_hidden_dim, args.decoder_hidden_dim, args.beta, args.gamma, args.dropout_rate).to(args.device)

vae.load_state_dict(torch.load('../model/cell_tokenzier/model.pth')['model'])
vae.eval() 

cuda


VQVAE(
  (encoder): Encoder(
    (network): Sequential(
      (0): Linear(in_features=5000, out_features=1600, bias=True)
      (1): LayerNorm((1600,), eps=1e-05, elementwise_affine=True)
      (2): PReLU(num_parameters=1)
      (3): Dropout(p=0.2, inplace=False)
      (4): Linear(in_features=1600, out_features=1024, bias=True)
      (5): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (6): PReLU(num_parameters=1)
      (7): Dropout(p=0.2, inplace=False)
      (8): Linear(in_features=1024, out_features=800, bias=True)
      (9): LayerNorm((800,), eps=1e-05, elementwise_affine=True)
      (10): PReLU(num_parameters=1)
      (11): Dropout(p=0.2, inplace=False)
      (12): Linear(in_features=800, out_features=256, bias=True)
    )
  )
  (vector_quantization): VectorQuantizer(
    (embedding): Embedding(256, 8)
  )
  (decoder): Decoder(
    (network): Sequential(
      (0): Linear(in_features=256, out_features=800, bias=True)
      (1): LayerNorm((800,), eps=1e-05, elementwise

#### Prepare Model and Tokenizer
Now, we will be adding 258 new tokens as well as replace the existing pad, bos and eos tokens of the model.

In [4]:
# Define new tokens
new_tokens = [f'<code_{i}>' for i in range(256)] + ['<cell_token_start>', '<cell_token_end>']

In [5]:
model_name = "your path to the LLM backbone"  # load the base LLM backbone
tokenizer_path = "your path to LLM's tokenizer"  # load the LLM toknizer

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_path,
    pad_token="<|pad|>",
    eos_token="<|endsentence|>",
    additional_special_tokens=new_tokens,
)

In [6]:
len(tokenizer)

151925

#### Load the processed dataset.

In [7]:
from datasets import load_dataset, load_from_disk
from tqdm import tqdm

versions = ["v2", "v3"]
donors = ["C55ctr", "C52ctr", "L22cov", "L17cov"]
version = versions[1]
donor = donors[0]

processed_datasets = load_from_disk("your path to CCC dataset")

In [8]:
processed_datasets

Dataset({
    features: ['index_name', 'i', 'code', 'story', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 13480
})

In [9]:
eval_subset = processed_datasets
eval_subset.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
eval_subset = eval_subset.shuffle(seed=42).select(range(100))
eval_subset

Dataset({
    features: ['index_name', 'i', 'code', 'story', 'input_text', 'output_text', 'input_ids', 'attention_mask', 'labels'],
    num_rows: 100
})

#### Load the pre-trained CellTok model

In [10]:
from peft import PeftModel

inference_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    low_cpu_mem_usage=True,
)
inference_model.resize_token_embeddings(len(tokenizer))

inference_model = PeftModel.from_pretrained(inference_model, "your path to CellTok-base")

inference_model.to("cuda")
inference_model.eval()

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

PeftModel(
  (base_model): LoraModel(
    (model): Qwen2ForCausalLM(
      (model): Qwen2Model(
        (embed_tokens): lora.Embedding(
          (base_layer): Embedding(151925, 3584)
          (lora_dropout): ModuleDict(
            (default): Identity()
          )
          (lora_A): ModuleDict()
          (lora_B): ModuleDict()
          (lora_embedding_A): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 64x151925 (cuda:0)])
          (lora_embedding_B): ParameterDict(  (default): Parameter containing: [torch.cuda.FloatTensor of size 3584x64 (cuda:0)])
          (lora_magnitude_vector): ModuleDict()
        )
        (layers): ModuleList(
          (0-27): 28 x Qwen2DecoderLayer(
            (self_attn): Qwen2SdpaAttention(
              (q_proj): lora.Linear(
                (base_layer): Linear(in_features=3584, out_features=3584, bias=True)
                (lora_dropout): ModuleDict(
                  (default): Identity()
                )
     

#### Evaluation

In [ ]:
# not given cell type
if version=="v2":
    import re

    def extract_cell_types_v2(text):
        match = re.search(r'this is a group of (.*?) and (.*?), and their communication', text, re.IGNORECASE)
        if match:
            cell_type1 = match.group(1).strip()
            cell_type2 = match.group(2).strip()
            return [cell_type1, cell_type2]
        return []
    
    def extract_pathways_v2(text):
        match = re.search(r'primarily occurs through the (.*?)(?= pathways)', text, re.IGNORECASE)
        if match:
            pathways = match.group(1).strip().rstrip('.').split(', ')
            pathways = [re.sub(r'[^a-zA-Z0-9]', '', pathway) for pathway in pathways]
            return pathways
        return []
    
    def evaluate_cell_types_and_pathways(target, target_predicted):
        target_cell_types = extract_cell_types_v2(target)
        predicted_cell_types = extract_cell_types_v2(target_predicted)
    
        if len(target_cell_types) == len(predicted_cell_types):
            cell_types_correct = all(t == p for t, p in zip(target_cell_types, predicted_cell_types))
        else:
            cell_types_correct = False
    
        target_pathways = extract_pathways_v2(target)
        predicted_pathways = extract_pathways_v2(target_predicted)
    
        target_set = set(target_pathways)
        predicted_set = set(predicted_pathways)
        
        correct_matches = target_set & predicted_set
        tp = len(correct_matches)  # True positives
        
        precision = tp / len(predicted_set) if len(predicted_set) > 0 else 0
        recall = tp / len(target_set) if len(target_set) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
        return {
            "cell_types_correct": cell_types_correct,
            "pathway_contains_correct": predicted_set.issubset(target_set),
            "pathway_exact_match": target_set == predicted_set,
            "pathway_precision": round(precision, 4), 
            "pathway_recall": round(recall, 4),
            "pathway_f1": round(f1, 4),
            "target_pathways": target_pathways,
            "predicted_pathways": predicted_pathways
        }

# given cell type
elif version == "v3":
    import re

    def extract_pathways_v2(text):
        match = re.search(r'primarily occurs through the (.*?)(?= pathways)', text, re.IGNORECASE)
        if match:
            pathways = match.group(1).strip().rstrip('.').split(', ')
            pathways = [re.sub(r'[^a-zA-Z0-9]', '', pathway) for pathway in pathways]
            return pathways
        return []
    
    def evaluate_cell_types_and_pathways(target, target_predicted):
        target_pathways = extract_pathways_v2(target)
        predicted_pathways = extract_pathways_v2(target_predicted)
    
        target_set = set(target_pathways)
        predicted_set = set(predicted_pathways)
        
        correct_matches = target_set & predicted_set
        tp = len(correct_matches)  # True positives
        
        precision = tp / len(predicted_set) if len(predicted_set) > 0 else 0
        recall = tp / len(target_set) if len(target_set) > 0 else 0
        f1 = 2 * (precision * recall) / (precision + recall) if (precision + recall) > 0 else 0
    
        return {
            "pathway_contains_correct": predicted_set.issubset(target_set),
            "pathway_exact_match": target_set == predicted_set,
            "pathway_precision": round(precision, 4), 
            "pathway_recall": round(recall, 4), 
            "pathway_f1": round(f1, 4),
            "target_pathways": target_pathways,
            "predicted_pathways": predicted_pathways
        }

In [ ]:
results = []
results_original = []

for sample in eval_subset:
    sample = {k: v.unsqueeze(0) for k, v in sample.items()} 
    sample = {k: v.to("cuda") for k, v in sample.items()}
    outputs = inference_model(**sample)
    
    valid_label_positions = (sample['labels'] == -100) & (sample['attention_mask'] == 1)
    filtered_input_ids = sample['input_ids'][valid_label_positions]
    output_tokens = inference_model.generate(
        filtered_input_ids.unsqueeze(0),
        max_new_tokens=1000,
        do_sample=True,
        temperature=0.2,
        top_p=0.95,
        top_k=50,
        eos_token_id=tokenizer.eos_token_id,
        pad_token_id=tokenizer.pad_token_id,
    )
    context = tokenizer.decode(filtered_input_ids, skip_special_tokens=False)

    target_predicted = tokenizer.decode(output_tokens[0], skip_special_tokens=True)

    labels = sample["labels"][0].cpu().tolist()
    valid_labels = [token for token in labels if token != -100]
    target = tokenizer.decode(valid_labels, skip_special_tokens=False)

    eval_result = evaluate_cell_types_and_pathways(target, target_predicted)
    results.append(eval_result)
    results_original.append(target_predicted)

if version=="v2":
    cell_types_correct = sum([result["cell_types_correct"] for result in results]) / len(results)
pathway_contains_correct = sum([result["pathway_contains_correct"] for result in results]) / len(results)
pathway_exact_match = sum([result["pathway_exact_match"] for result in results]) / len(results)
pathway_precision = sum([result["pathway_precision"] for result in results]) / len(results)
pathway_recall = sum([result["pathway_recall"] for result in results]) / len(results)
pathway_f1 = sum([result["pathway_f1"] for result in results]) / len(results)

if version=="v2":
    print(f"Cell Type Accuracy: {cell_types_correct:.2f}")
print(f"Pathway Containment Accuracy: {pathway_contains_correct:.4f}")
print(f"Pathway Exact Match Accuracy: {pathway_exact_match:.4f}")
print(f"Pathway Precision: {pathway_precision:.4f}")
print(f"Pathway Recall: {pathway_recall:.4f}")
print(f"Pathway F1: {pathway_f1:.4f}")